[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day10_feature_selection/day10_notebook.ipynb)

# Day 10 / 42: Feature Selection
### 42 Days of AI/ML Challenge | #42DaysOfML

---

## What You Will Learn
- Why more features does not mean a better model
- **Filter methods:** Correlation, Chi-Square, Mutual Information
- **Wrapper methods:** Recursive Feature Elimination (RFE)
- **Embedded methods:** Random Forest importance, Lasso regularization
- How to compare which features each method selects
- Before vs After model accuracy on a real dataset
- 3 practice exercises with solutions

**Concept:** Feature selection is the process of removing features that add noise, redundancy, or no predictive signal so your model trains faster, generalizes better, and is easier to interpret.

> **Prerequisites:** Day 9 (Feature Engineering). You should know how to create features. Today you learn which ones to keep.


## Why Feature Selection Matters

Adding more features feels like giving your model more information. But irrelevant features:
- Increase training time (sometimes by 10x for large datasets)
- Introduce noise that confuses the model
- Cause overfitting — the model memorizes noise instead of learning patterns
- Make the model harder to explain to stakeholders

**Meta's ad click model** removed 40% of low-variance features before training, cutting training time in half while keeping the same prediction accuracy. The removed features weren't wrong — they just weren't useful.

There are 3 families of feature selection methods. Each has different strengths.

## Setup

In [ ]:
# Uncomment if running in Colab
# !pip install pandas numpy scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif,
    RFE, SelectFromModel,
    mutual_info_classif
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")
print(f"scikit-learn version: {__import__('sklearn').__version__}")

---

## The Dataset: MLE Job Hiring Prediction

A dataset of 3,000 engineering candidates. Some features are meaningful (GPA, LeetCode solved, experience). Some are pure noise (random_id, zip_code, lucky_number).

**Goal:** Predict whether a candidate gets hired (1) or not (0).

The challenge: your dataset has 15 features. Some help. Some hurt. Feature selection finds which is which.

In [ ]:
np.random.seed(42)
n = 3000

df = pd.DataFrame({
    # Meaningful features
    'age':               np.random.randint(22, 60, n),
    'experience_years':  np.random.randint(0, 35, n),
    'num_projects':      np.random.randint(1, 20, n),
    'github_stars':      np.random.randint(0, 5000, n),
    'leetcode_solved':   np.random.randint(0, 600, n),
    'certifications':    np.random.randint(0, 10, n),
    'interview_rounds':  np.random.randint(1, 8, n),
    'gpa':               np.round(np.random.uniform(5.0, 10.0, n), 2),
    'company_tier':      np.random.choice([1, 2, 3], n),
    'skills_count':      np.random.randint(3, 30, n),
    # Pure noise features
    'random_id':         np.random.randint(1000, 9999, n),
    'zip_code':          np.random.randint(100000, 999999, n),
    'lucky_number':      np.random.randint(1, 100, n),
    'noise_1':           np.random.randn(n),
    'noise_2':           np.random.randn(n),
})

# Target: hired based on real rules (with 5% random noise)
df['hired'] = (
    (df['experience_years'] > 2) &
    (df['leetcode_solved'] > 100) &
    (df['gpa'] > 6.5)
).astype(int)

flip_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
df.loc[flip_idx, 'hired'] = 1 - df.loc[flip_idx, 'hired']

print(f"Dataset shape: {df.shape}")
print(f"\nHired distribution:")
print(df['hired'].value_counts())
print(f"\nHire rate: {df['hired'].mean():.1%}")

# Split early — before any feature selection
X = df.drop('hired', axis=1)
y = df['hired']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain size: {X_train.shape}, Test size: {X_test.shape}")
print(f"All 15 features: {X.columns.tolist()}")

### Baseline: All 15 Features

In [ ]:
rf_all = RandomForestClassifier(n_estimators=100, random_state=42)
rf_all.fit(X_train, y_train)
acc_all = accuracy_score(y_test, rf_all.predict(X_test))

print(f"Baseline accuracy using ALL 15 features: {acc_all:.4f} ({acc_all*100:.2f}%)")
print("We'll compare this against models using only selected features.")

---

## METHOD 1: Filter Methods

Filter methods evaluate each feature **independently** using a statistical score. They do not involve any model. Fast, scalable, good for a first pass.

Three filter methods worth knowing:
- **Correlation:** Linear relationship between feature and target
- **Chi-Square:** Association between categorical/non-negative features and target
- **Mutual Information:** Captures non-linear relationships too

### Filter Method 1A: Correlation with Target

In [ ]:
# Compute absolute correlation of each feature with target
# Use training data only
train_df = X_train.copy()
train_df['hired'] = y_train.values

corr_with_target = (
    train_df.drop('hired', axis=1)
    .corrwith(train_df['hired'])
    .abs()
    .sort_values(ascending=False)
)

print("Absolute correlation with target:")
print(corr_with_target.round(4).to_string())

# Visualise
plt.figure(figsize=(9, 5))
colors = ['#00C853' if v > 0.1 else '#E53935' if v < 0.02 else '#1565C0'
          for v in corr_with_target]
corr_with_target.plot(kind='bar', color=colors, edgecolor='white')
plt.axhline(0.1, color='green', linestyle='--', linewidth=1, label='High signal (>0.1)')
plt.axhline(0.02, color='red', linestyle='--', linewidth=1, label='Low signal (<0.02)')
plt.title('Correlation of Each Feature with Target (Hired)', fontweight='bold')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

# Threshold: keep features with correlation > 0.05
corr_selected = corr_with_target[corr_with_target > 0.05].index.tolist()
print(f"\nFeatures selected (correlation > 0.05): {corr_selected}")
print(f"Removed as low-signal: {[f for f in X.columns if f not in corr_selected]}")

### Filter Method 1B: Chi-Square Test

Chi-square tests whether a feature and the target are statistically independent. A high chi-square score = the feature and target are NOT independent = the feature carries signal.

**Requirement:** All values must be non-negative. Apply MinMaxScaler first.

In [ ]:
# Scale to [0,1] — chi2 requires non-negative values
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use train scaler on test

chi2_scores, chi2_pvals = chi2(X_train_scaled, y_train)

chi2_df = pd.DataFrame({
    'feature':   X.columns,
    'chi2_score': chi2_scores,
    'p_value':    chi2_pvals
}).sort_values('chi2_score', ascending=False).reset_index(drop=True)

print("Chi-Square scores:")
print(chi2_df.round(4).to_string())

# Features with p_value < 0.05 are statistically significant
chi2_selected = chi2_df[chi2_df['p_value'] < 0.05]['feature'].tolist()
print(f"\nStatistically significant features (p < 0.05): {chi2_selected}")
print(f"Not significant (p >= 0.05): {chi2_df[chi2_df['p_value'] >= 0.05]['feature'].tolist()}")

# Visualise p-values
plt.figure(figsize=(9, 4))
colors = ['#00C853' if p < 0.05 else '#E53935' for p in chi2_df['p_value']]
plt.bar(chi2_df['feature'], -np.log10(chi2_df['p_value'] + 1e-10), color=colors, edgecolor='white')
plt.axhline(-np.log10(0.05), color='orange', linestyle='--', label='p=0.05 threshold')
plt.title('Chi-Square: -log10(p-value) per Feature', fontweight='bold')
plt.ylabel('-log10(p-value)  [higher = more significant]')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

### Filter Method 1C: Mutual Information

Mutual Information measures how much knowing a feature reduces uncertainty about the target. Unlike correlation, it captures **non-linear** relationships too.

A mutual information score of 0 means the feature and target are completely independent.

In [ ]:
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    'feature':  X.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False).reset_index(drop=True)

print("Mutual Information scores:")
print(mi_df.round(4).to_string())

mi_selected = mi_df[mi_df['mi_score'] > 0.01]['feature'].tolist()
print(f"\nSelected (MI > 0.01): {mi_selected}")

plt.figure(figsize=(9, 4))
colors = ['#00C853' if v > 0.05 else '#1565C0' if v > 0.01 else '#E53935'
          for v in mi_df['mi_score']]
plt.bar(mi_df['feature'], mi_df['mi_score'], color=colors, edgecolor='white')
plt.axhline(0.05, color='green', linestyle='--', linewidth=1, label='Strong signal (>0.05)')
plt.axhline(0.01, color='orange', linestyle='--', linewidth=1, label='Weak signal (>0.01)')
plt.title('Mutual Information Score per Feature', fontweight='bold')
plt.ylabel('MI Score')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

### Compare All 3 Filter Methods

In [ ]:
comparison = pd.DataFrame(index=X.columns)
comparison['correlation_selected'] = comparison.index.isin(corr_selected).astype(int)
comparison['chi2_selected']        = comparison.index.isin(chi2_selected).astype(int)
comparison['mi_selected']          = comparison.index.isin(mi_selected).astype(int)
comparison['votes']                = comparison.sum(axis=1)
comparison = comparison.sort_values('votes', ascending=False)

print("Feature selection comparison (1 = selected, 0 = rejected):")
print(comparison.to_string())

# Features selected by ALL 3 methods = most reliable
agreed_by_all = comparison[comparison['votes'] == 3].index.tolist()
print(f"\nSelected by ALL 3 filter methods: {agreed_by_all}")
print("These are your highest-confidence features from filter-based selection.")

---

## METHOD 2: Wrapper Methods — Recursive Feature Elimination (RFE)

Wrapper methods train a model, rank features by importance, remove the weakest ones, then retrain. They repeat this loop until you reach the number of features you want.

**Strength:** Captures feature interactions that filter methods miss.  
**Weakness:** Computationally expensive. For datasets with 1000+ features, use filter methods first to reduce the search space, then apply RFE.

Sklearn's `RFE` does exactly this loop automatically.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr, n_features_to_select=6, step=1)
rfe.fit(X_train_scaled, y_train)  # use scaled data for logistic regression

rfe_features  = X.columns[rfe.support_].tolist()
rfe_ranking   = pd.Series(rfe.ranking_, index=X.columns).sort_values()

print("RFE feature ranking (1 = selected, higher = eliminated earlier):")
print(rfe_ranking.to_string())
print(f"\nRFE selected 6 features: {rfe_features}")

# Visualise
plt.figure(figsize=(9, 4))
colors = ['#00C853' if r == 1 else '#E53935' for r in rfe_ranking]
rfe_ranking.plot(kind='bar', color=colors, edgecolor='white')
plt.title('RFE Feature Ranking (rank=1 means selected)', fontweight='bold')
plt.ylabel('Rank (lower is better)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## METHOD 3: Embedded Methods

Embedded methods perform feature selection **during** model training. The model assigns importance to each feature as part of its learning process.

Two most common:
- **Random Forest feature importance:** measures how much each feature reduces impurity across all trees
- **Lasso regularization:** shrinks coefficients of irrelevant features to exactly 0, effectively removing them

### Embedded Method 3A: Random Forest Feature Importance

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Random Forest feature importances:")
print(rf_importance.round(4).to_string())

# Select features above mean importance
mean_importance = rf_importance.mean()
rf_selected = rf_importance[rf_importance > mean_importance].index.tolist()
print(f"\nMean importance threshold: {mean_importance:.4f}")
print(f"RF selected features: {rf_selected}")

# Visualise
plt.figure(figsize=(9, 5))
colors = ['#00C853' if imp > mean_importance else '#E53935'
          for imp in rf_importance]
rf_importance.plot(kind='bar', color=colors, edgecolor='white')
plt.axhline(mean_importance, color='orange', linestyle='--',
            linewidth=1.5, label=f'Mean importance ({mean_importance:.4f})')
plt.title('Random Forest Feature Importance', fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

### Embedded Method 3B: Lasso Regularization

Lasso adds a penalty term to the loss function that forces small coefficients toward exactly 0. Features with coefficient = 0 are automatically removed.

`LassoCV` finds the best regularization strength automatically using cross-validation.

In [ ]:
lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

lasso_coef = pd.Series(
    np.abs(lasso.coef_),
    index=X.columns
).sort_values(ascending=False)

print(f"Optimal Lasso alpha (regularization strength): {lasso.alpha_:.6f}")
print("\nLasso absolute coefficients:")
print(lasso_coef.round(6).to_string())

lasso_selected = lasso_coef[lasso_coef > 0].index.tolist()
lasso_removed  = lasso_coef[lasso_coef == 0].index.tolist()
print(f"\nLasso kept (coef > 0): {lasso_selected}")
print(f"Lasso zeroed out (removed): {lasso_removed}")

plt.figure(figsize=(9, 4))
colors = ['#00C853' if v > 0 else '#E53935' for v in lasso_coef]
lasso_coef.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Lasso Coefficients (zero = feature removed)', fontweight='bold')
plt.ylabel('Absolute Coefficient')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## Full Comparison: All 5 Methods Side by Side

In [ ]:
full_comparison = pd.DataFrame(index=X.columns)
full_comparison['Correlation']  = full_comparison.index.isin(corr_selected).astype(int)
full_comparison['Chi-Square']   = full_comparison.index.isin(chi2_selected).astype(int)
full_comparison['Mutual Info']  = full_comparison.index.isin(mi_selected).astype(int)
full_comparison['RFE']          = full_comparison.index.isin(rfe_features).astype(int)
full_comparison['Random Forest']= full_comparison.index.isin(rf_selected).astype(int)
full_comparison['Lasso']        = full_comparison.index.isin(lasso_selected).astype(int)
full_comparison['Total Votes']  = full_comparison.sum(axis=1)
full_comparison = full_comparison.sort_values('Total Votes', ascending=False)

print("Method comparison (1=selected, 0=rejected):")
print(full_comparison.to_string())

# Heatmap
plt.figure(figsize=(11, 6))
heatmap_data = full_comparison.drop('Total Votes', axis=1)
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='RdYlGn',
            linewidths=0.5, cbar=False,
            xticklabels=heatmap_data.columns,
            yticklabels=heatmap_data.index)
plt.title('Feature Selection: Which Method Picks Which Feature', fontweight='bold')
plt.xlabel('Method')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

consensus_features = full_comparison[full_comparison['Total Votes'] >= 4].index.tolist()
print(f"\nConsensus features (selected by 4+ methods): {consensus_features}")

---

## Before vs After: Model Accuracy

In [ ]:
results = {}

def evaluate(name, features, use_scaled=False):
    X_tr = X_train_scaled[:, [list(X.columns).index(f) for f in features]] if use_scaled else X_train[features]
    X_te = X_test_scaled[:, [list(X.columns).index(f) for f in features]]  if use_scaled else X_test[features]
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_tr, y_train)
    acc = accuracy_score(y_test, rf.predict(X_te))
    results[name] = {'features': len(features), 'accuracy': acc}
    return acc

acc_all      = evaluate('All 15 Features',              list(X.columns))
acc_corr     = evaluate('Filter: Correlation',          corr_selected)
acc_chi2     = evaluate('Filter: Chi-Square',           chi2_selected)
acc_mi       = evaluate('Filter: Mutual Info',          mi_selected)
acc_rfe      = evaluate('Wrapper: RFE',                 rfe_features, use_scaled=True)
acc_rf       = evaluate('Embedded: Random Forest',      rf_selected)
acc_lasso    = evaluate('Embedded: Lasso',              lasso_selected if lasso_selected else list(X.columns))
acc_consensus= evaluate('Consensus (4+ votes)',         consensus_features)

results_df = pd.DataFrame(results).T.sort_values('accuracy', ascending=False)
results_df['accuracy_pct'] = (results_df['accuracy'] * 100).round(2)
print("Results summary:")
print(results_df[['features','accuracy_pct']].to_string())

plt.figure(figsize=(10, 5))
bars = plt.barh(
    results_df.index,
    results_df['accuracy'],
    color=['#E53935' if n == 'All 15 Features' else '#00C853'
           if v == results_df['accuracy'].max() else '#1565C0'
           for n, v in zip(results_df.index, results_df['accuracy'])],
    edgecolor='white'
)
for bar, (name, row) in zip(bars, results_df.iterrows()):
    plt.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2,
             f"{row['accuracy_pct']:.2f}%  ({int(row['features'])} features)",
             va='center', ha='right', color='white', fontweight='bold', fontsize=9)
plt.xlim(0.88, 1.01)
plt.title('Accuracy vs Feature Selection Method', fontweight='bold')
plt.xlabel('Accuracy')
plt.tight_layout()
plt.show()

---

## Real World Problem: The High-Cardinality Trap

Notice that `zip_code` appeared as important in the Random Forest importance scores even though it is pure noise.

This is a documented trap called the **high-cardinality bias** in tree-based feature importance. Features with many unique values (like zip codes, IDs, timestamps as integers) get artificially inflated importance scores because trees can split on them in many ways.

**The production fix:** Use permutation importance instead of impurity-based importance for tree models. Permutation importance directly measures how much accuracy drops when a feature is randomly shuffled, which is immune to high-cardinality bias.

In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf_all, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_importance = pd.Series(
    perm_result.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print("Permutation importance (immune to high-cardinality bias):")
print(perm_importance.round(4).to_string())

# Compare: impurity-based vs permutation for zip_code
print(f"\nzip_code impurity importance: {rf_importance['zip_code']:.4f}")
print(f"zip_code permutation importance: {perm_importance['zip_code']:.4f}")
print("If permutation is near 0 but impurity was high: high-cardinality bias confirmed.")

# Visualise comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rf_importance.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Impurity-Based Importance\n(biased for high-cardinality features)', fontweight='bold')
axes[0].set_xlabel('Score')

perm_importance.sort_values().plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Permutation Importance\n(unbiased, production-safe)', fontweight='bold')
axes[1].set_xlabel('Score')

plt.suptitle('Impurity vs Permutation Importance: Spot the Difference', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary: Which Method to Use When

| Method | Speed | Captures Interactions | Best For |
|--------|-------|----------------------|----------|
| Correlation | Very fast | No | First-pass filter, linear relationships |
| Chi-Square | Very fast | No | Categorical features, classification |
| Mutual Information | Fast | Yes | Non-linear relationships |
| RFE | Slow | Yes | Small-medium datasets, final selection |
| RF Importance | Medium | Yes | Tree-based models, quick embedded selection |
| Lasso | Medium | No | Linear models, sparse selection |

**Production workflow:**  
1. Run filter methods first to remove obvious noise quickly  
2. Run RFE or embedded methods on the reduced set  
3. Use permutation importance to validate final selections  
4. Always evaluate accuracy before vs after on held-out test data

---

## Practice Exercises

In [ ]:
# EXERCISE 1
# Use SelectKBest with f_classif (ANOVA F-test) to select the top 5 features.
# Print which features were selected.
# Train a Random Forest on these 5 features and report accuracy.

# Hint: SelectKBest(score_func=f_classif, k=5)

# Your code here:


In [ ]:
# EXERCISE 2
# Run RFE with n_features_to_select = 3, 5, 7, and 10.
# Plot accuracy vs number of features selected.
# At what point does adding more features stop helping?

# Your code here:


In [ ]:
# EXERCISE 3 (Challenge)
# Variance Threshold is the simplest filter method: remove features with very low variance.
# Features with near-zero variance carry almost no information.
# Use sklearn's VarianceThreshold with threshold=0.1 on the scaled dataset.
# Which features get removed? Does this match what the other methods found?

# Hint: from sklearn.feature_selection import VarianceThreshold

# Your code here:


---

## Solutions

In [ ]:
# SOLUTION 1: SelectKBest with ANOVA F-test
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X_train, y_train)

kbest_features = X.columns[selector.get_support()].tolist()
print(f"SelectKBest top 5 features: {kbest_features}")

X_train_kb = selector.transform(X_train)
X_test_kb  = selector.transform(X_test)

rf_kb = RandomForestClassifier(n_estimators=100, random_state=42)
rf_kb.fit(X_train_kb, y_train)
acc_kb = accuracy_score(y_test, rf_kb.predict(X_test_kb))
print(f"Accuracy with top 5 features: {acc_kb:.4f} ({acc_kb*100:.2f}%)")
print(f"Baseline (all 15): {acc_all:.4f}")

In [ ]:
# SOLUTION 2: RFE accuracy vs number of features
k_values = [3, 5, 7, 10]
rfe_accuracies = []

for k in k_values:
    rfe_k = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42),
                n_features_to_select=k, step=1)
    rfe_k.fit(X_train_scaled, y_train)
    feats = X.columns[rfe_k.support_].tolist()

    rf_k = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_k.fit(X_train[feats], y_train)
    acc_k = accuracy_score(y_test, rf_k.predict(X_test[feats]))
    rfe_accuracies.append(acc_k)
    print(f"k={k}: accuracy={acc_k:.4f}  features={feats}")

plt.figure(figsize=(7, 4))
plt.plot(k_values, rfe_accuracies, marker='o', color='steelblue', linewidth=2)
plt.axhline(acc_all, color='red', linestyle='--', label=f'All 15 features ({acc_all:.4f})')
plt.xlabel('Number of Features Selected (k)')
plt.ylabel('Accuracy')
plt.title('RFE: Accuracy vs Number of Features', fontweight='bold')
plt.xticks(k_values)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION 3: VarianceThreshold
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.1)
vt.fit(X_train_scaled)

vt_kept    = X.columns[vt.get_support()].tolist()
vt_removed = X.columns[~vt.get_support()].tolist()

print(f"Features KEPT by VarianceThreshold: {vt_kept}")
print(f"Features REMOVED (low variance): {vt_removed}")

variances = pd.Series(vt.variances_, index=X.columns).sort_values()
print("\nVariance per feature:")
print(variances.round(4).to_string())

---

## What's Next

**Day 11: Scaling and Encoding**  
You now know which features to keep. Tomorrow you learn how to prepare them correctly before feeding into a model — MinMax vs StandardScaler, Label Encoding vs One-Hot Encoding, and what happens when you skip this step.

---

**GitHub repo:** https://github.com/VaishnaviJagtap18/42-days-aiml-challenge  
**Follow along:** #42DaysOfML